# 📊 Analyse Exploratoire des Tweets
Notebook d'exploration des données et des modèles de sentiment.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

df = pd.read_csv('../data/tweets.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Distribution des sentiments
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c', '#3498db'])
axes[0].set_title('Distribution des sentiments')
axes[0].set_ylabel('Nombre de tweets')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c', '#3498db'])
axes[1].set_title('Répartition en pourcentage')

plt.tight_layout()
plt.show()

In [ ]:
# Longueur des tweets
df['text_length'] = df['text'].apply(len)
df['word_count']  = df['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for sentiment, color in zip(['positive','negative','neutral'], ['#2ecc71','#e74c3c','#3498db']):
    subset = df[df['sentiment'] == sentiment]
    axes[0].hist(subset['text_length'], alpha=0.6, label=sentiment, color=color, bins=20)
    axes[1].hist(subset['word_count'],  alpha=0.6, label=sentiment, color=color, bins=15)

axes[0].set_title('Distribution de la longueur des tweets')
axes[0].legend()
axes[1].set_title('Distribution du nombre de mots')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Entraîner un modèle rapide
import sys
sys.path.insert(0, '..')
from ml.train_model import preprocess, load_config
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df['text_clean'] = df['text'].apply(preprocess)
X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_v = vectorizer.fit_transform(X_train)
X_test_v  = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=500)
model.fit(X_train_v, y_train)
y_pred = model.predict(X_test_v)
print(classification_report(y_test, y_pred))

In [1]:
# Test prédiction
tests = [
    'I love this product, amazing experience!',
    'Terrible service, very disappointed',
    'The meeting is at 3pm tomorrow',
]
for tweet in tests:
    cleaned  = preprocess(tweet)
    features = vectorizer.transform([cleaned])
    pred     = model.predict(features)[0]
    print(f'{pred:10s} | {tweet}')

NameError: name 'preprocess' is not defined